# Major Montana landowners FINAL analysis



In [1]:
# Library import and config
import json
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_rows', 100) # Avoids truncation of long data table results
pd.set_option('display.max_columns', 100) # Avoids truncation of wide data table results

In [2]:
# Read in data (slow)
# Parcel data from **March 9** download -- may need to update file path
parcels = gpd.read_file('./raw/MontanaCadastral_SHP_3-9-2026/Montana_Cadastral/OWNERPARCEL.shp').to_crs(epsg=4326)



In [3]:
# Add in ownership groupings

with open('./owner-name-grouping.json', 'r') as file:
    OWNER_GROUPS_RAW = json.load(file)

cleaner = {}
for group in OWNER_GROUPS_RAW:
    ownerCleaned = group['owner']
    for ownerAlternate in group['alternatesOwnershipNames']:
        cleaner[ownerAlternate] = ownerCleaned
parcels['OwnerName_Grouped'] = parcels['OwnerName'].replace(cleaner)



In [5]:
# Group parcels by owner name, count the number in each group and sum by acreage in each group
# Running more than top 10 here because most of the biggest are public entities, which we need to filter out
top_100_landowners = parcels.groupby(['OwnerName_Grouped']).agg({
    'PARCELID': 'count',
    'TotalAcres': 'sum',
    'OwnerName': 'unique',
}).sort_values('TotalAcres', ascending=False).head(100)

top_100_landowners.head(10) # Show top 10 to check output

,PARCELID,TotalAcres,OwnerName
OwnerName_Grouped,,,
USDA FOREST SERVICE,16672,8775511.897,[USDA FOREST SERVICE]
USDI BUREAU OF LAND MANAGEMENT,18102,5604489.400,[USDI BUREAU OF LAND MANAGEMENT]
STATE OF MONTANA,12223,4265228.436,[STATE OF MONTANA]
UNITED STATES OF AMERICA,5580,1881103.943,[UNITED STATES OF AMERICA]
USA,3985,1828458.927,[USA]
USA IN TRUST FOR CROW TRIBE,3848,1303624.520,[USA IN TRUST FOR CROW TRIBE]
USA IN TRUST,5279,1139955.995,[USA IN TRUST]
USA - FOREST SERVICE,2161,1115836.824,[USA - FOREST SERVICE]
BUREAU OF LAND MANAGEMENT,2735,732067.380,[BUREAU OF LAND MANAGEMENT]


In [6]:
# Filter out manually identified public landowners

with open('./known-public-landowners.json', 'r') as file:
    PUBLIC_LANDOWNERS = json.load(file)

top_100_private_only = top_100_landowners[~top_100_landowners.index.isin(PUBLIC_LANDOWNERS)]

# Add rank for display purposes
ranking = top_100_private_only.copy().reset_index()
ranking['Rank'] = ranking.index+1
ranking.rename(columns={
    'PARCELID': 'NumParcels',
    'OwnerName_Grouped': 'Owner',
    'OwnerName': 'IncludedOwnerNames',
    }, inplace=True)
ranking.head(30)[['Rank','Owner','NumParcels','TotalAcres',]] # show top 20

,Rank,Owner,NumParcels,TotalAcres
0,1,GALT RANCH,820,346571.264
1,2,FARMLAND RESERVE INC,653,316931.462
2,3,BROKEN O LAND & LIVESTOCK LLC,780,291870.037
3,4,GREEN DIAMOND RESOURCE COMPANY,644,291802.938
4,5,WILKS BROTHERS,705,275634.288
5,6,COFFEE-NEFSY,503,238831.067
6,7,SUNLIGHT RANCH COMPANY,713,236713.603
7,8,AMERICAN PRAIRIE,606,145969.964
8,9,NATURE CONSERVANCY,389,142986.131
9,10,GREAT NORTHERN PROPERTIES,292,142849.817


In [7]:
# Write top 20 list to outputs file as JSON
ranking.head(20).to_json('./outputs/final-top-20-list.json', orient='records', indent=4, index=False)

In [8]:
# More human-readable listing with acreages for each underlying owner name
# Manually copy and paste to outputs/
    

for idx, row in ranking.head(20).iterrows():
    print('#' + str(idx+1) + ' / ' + row['Owner'])
    print(f'{row['TotalAcres']:,.0f} acres - {str(row['NumParcels'])} parcels')
    print('  Includes:')
    for subOwner in row['IncludedOwnerNames']:
        subOwnerParcels = parcels[parcels['OwnerName'] == subOwner]
        print(f'    {subOwner} --> {subOwnerParcels['TotalAcres'].sum():,.0f} acres/ {len(subOwnerParcels)} parcels')
    print('\n')

#1 / GALT RANCH
346,571 acres - 820 parcels
  Includes:
    SUN COULEE LLC --> 4,508 acres/ 20 parcels
    71 RANCH LP --> 249,141 acres/ 616 parcels
    GALT RANCH LP --> 92,922 acres/ 182 parcels
    GALT ERROL T --> 1 acres/ 1 parcels
    GALT ERROL T & SHARRIE K --> 0 acres/ 1 parcels


#2 / FARMLAND RESERVE INC
316,931 acres - 653 parcels
  Includes:
    FARMLAND RESERVE, INC --> 18,197 acres/ 30 parcels
    FARMLAND RESERVE INC --> 298,535 acres/ 621 parcels
    FARMLAND RESERVES INC --> 200 acres/ 2 parcels


#3 / BROKEN O LAND & LIVESTOCK LLC
291,870 acres - 780 parcels
  Includes:
    PV RANCH COMPANY LLC --> 116,672 acres/ 212 parcels
    PV RANCH CO LLC --> 16,312 acres/ 32 parcels
    FORT PEASE COMM PASTURE --> 20,847 acres/ 41 parcels
    BROKEN O LAND & LIVESTOCK LLC --> 119,111 acres/ 454 parcels
    KROENKE LAND & LIVESTOCK LLC --> 12,485 acres/ 25 parcels
    KROENKE LAND & LIVESTOCK CO LLC --> 3,332 acres/ 10 parcels
    KROENKE LAND & LIVESTOCK --> 651 acres/ 2 parc

In [ ]:
# # Write parcels to separate file to allow analysis using MTFP-assigned groupings in explore.ipyb
parcels.to_file('./cleaned/parcels-with-mtfp-groupings.shp')

In [26]:
# Export geojson data for the parcels owned by each of the top 20 landowners 
majorColumns = ['CountyName','OwnerName_Grouped','OwnerName','OwnerAddre','OwnerAdd_1','OwnerAdd_2','OwnerCity','OwnerState','PropType','TotalAcres','geometry']
for idx, row in ranking.head(20).iterrows():
    subOwnerParcels = parcels[parcels['OwnerName_Grouped'] == row['Owner']][majorColumns]
    path = f'./geodata-outputs/{str(idx+1)}-{row['Owner'].replace(' ','-').replace('&-','')}.geojson'
    subOwnerParcels.to_file(path, driver="GeoJSON")